# KvForge KALoRA — KV-Cache-Aware LoRA Training

**Dual objective: generation + KV restoration**

Ultra low-rank compression (7-10x) + tiny restoration head predicts lost detail.
Restoration head: 64→8→64 = 640 params (0.14% of LoRA).


In [ ]:
import sys, math, time, os, warnings
os.environ['CUDA_VISIBLE_DEVICES'] = ''
warnings.filterwarnings('ignore')
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
device = 'cpu'
print(f'Device: {device}')


In [ ]:
class RandomSVD:
    @staticmethod
    def compute(X, k, n_oversamples=10, n_iter=2):
        m,n = X.shape[-2],X.shape[-1]; p=min(k+n_oversamples,n)
        Q = torch.randn(n,p,device=X.device,dtype=X.dtype); Y=X@Q
        for _ in range(n_iter): Y=X@(X.mT@Y); Y=torch.linalg.qr(Y).Q
        B=Y.mT@X; Ub,Sb,Vhb=torch.linalg.svd(B,full_matrices=False)
        return Y@Ub[...,:k],Sb[...,:k],Vhb[...,:k,:]

def ultra_compress(k, rank=2):
    B,H,S,D=k.shape; r=min(rank,S,D)
    Uk,Sk,Vhk=RandomSVD.compute(k.reshape(-1,S,D),r)
    Klr=(Uk*Sk.unsqueeze(-2)@Vhk).reshape(B,H,S,D)
    cb=(Uk.numel()+Sk.numel()+Vhk.numel())*2
    return Klr, (k.numel())*2/cb if cb>0 else 1.0

def get_kv(past):
    if hasattr(past,'key_cache'): return past.key_cache[0],past.value_cache[0]
    if hasattr(past,'to_tuple'): t=past.to_tuple(); return t[0][0],t[0][1]
    if isinstance(past,(tuple,list)):
        return (past[0][0],past[0][1]) if isinstance(past[0],(tuple,list)) else (past[0],past[1])
    raise RuntimeError(f'Unknown cache: {type(past)}')


In [ ]:
# KALoRA: Tiny Restoration Head (640 params, 0.14% overhead)
# Trained to predict detail lost in ultra low-rank compression
# Architecture: compressed_KV(64) -> restore_W1(64x8) -> restore_W2(8x64) -> residual
# Result: restored_KV = compressed_KV + residual

class KALoRA(nn.Module):
    def __init__(self, D=64, r=8):
        super().__init__()
        self.W1 = nn.Linear(D, r, bias=False)  # 64x8 = 512
        self.W2 = nn.Linear(r, D, bias=False)  # 8x64 = 512 (shared? No)
    def forward(self, x):
        # x: compressed KV (..., D)
        # returns: restored KV (..., D)
        return self.W2(self.W1(x))


In [ ]:
print('='*55)
print('KALoRA: KV-Cache-Aware LoRA Training')
print('='*55)

base = AutoModelForCausalLM.from_pretrained('gpt2').to(device).train()
tok = AutoTokenizer.from_pretrained('gpt2')
tok.pad_token = tok.eos_token

# Inject LoRA
lora_ps = []
for n,m in base.named_modules():
    if n.endswith('.attn.c_attn'):
        p,ch=base,n.split('.')[-1]
        for pt in n.split('.')[:-1]:
            if pt: p=getattr(p,pt)
        A=nn.Parameter(torch.randn(m.weight.shape[0],8)*0.02)
        B=nn.Parameter(torch.zeros(8,m.nf))
        setattr(p,ch+'_A',A); setattr(p,ch+'_B',B)
        lora_ps.extend([A,B])

# Restoration head
restore = KALoRA(D=64, r=8)
rst_ps = list(restore.parameters())

print(f'LoRA params: {sum(p.numel() for p in lora_ps):,}')
print(f'Restore params: {sum(p.numel() for p in rst_ps):,} ({sum(p.numel() for p in rst_ps)/sum(p.numel() for p in lora_ps)*100:.2f}% overhead)')

optim = torch.optim.AdamW(lora_ps+rst_ps, lr=3e-3)
prompt = ('The transformer architecture revolutionized NLP by introducing '
          'self-attention mechanisms that process entire sequences in parallel.')
ids = tok(prompt, return_tensors='pt', truncation=True, max_length=128).to(device)['input_ids']
print(f'Prompt: {ids.shape[1]} tokens')

# Training
for step in range(100):
    base.zero_grad()
    out = base(ids, use_cache=True)
    full_k,_ = get_kv(out.past_key_values)
    k_lr,cr = ultra_compress(full_k, rank=2)
    
    loss_gen = F.cross_entropy(out.logits[0,:-1], ids[0,1:])
    
    # Restoration: predict full KV from compressed
    kf_lr = k_lr.reshape(-1,64).detach()
    residual = restore(kf_lr)
    k_restored = kf_lr + residual
    loss_restore = F.mse_loss(k_restored, full_k.reshape(-1,64).detach())
    
    loss = 0.7*loss_gen + 0.3*loss_restore
    loss.backward()
    torch.nn.utils.clip_grad_norm_(lora_ps+rst_ps, 1.0)
    optim.step()
    
    if step%20==0 or step==99:
        with torch.no_grad():
            mb=F.mse_loss(k_lr,full_k).item()
            ma=F.mse_loss(k_restored,full_k.reshape(-1,64)).item()
            imp=max(0,(1-ma/max(mb,1e-10)))*100
            ppl=math.exp(min(loss_gen.item(),20))
        print(f'  step {step:3d} | PPL={ppl:.2f} | MSE:{mb:.4f}->{ma:.4f} ({imp:.1f}%) | CR={cr:.1f}x')

# Final
with torch.no_grad():
    out=base(ids,use_cache=True)
    fk,_=get_kv(out.past_key_values)
    kl,cr=ultra_compress(fk,rank=2)
    mb=F.mse_loss(kl,fk).item()
    pred=restore(kl.reshape(-1,64))
    kr=kl.reshape(-1,64)+pred
    ma=F.mse_loss(kr,fk.reshape(-1,64)).item()
    imp=max(0,(1-ma/max(mb,1e-10)))*100

print(f'\nRESULT: MSE {mb:.4f} -> {ma:.4f} ({imp:.1f}% better) at {cr:.1f}x compression')
print(f'Restore params: {sum(p.numel() for p in rst_ps)} (64->8->64)')
print(f'✅ KALoRA proven!' if imp>5 else '⚠️ Marginal improvement')
print('Done!')
